In [1]:
import os
import sys
import json
import random
from typing import Any, Union

import pandas as pd
import numpy as np
import jsonlines

sys.path.append(f"{os.environ['MDS']}/code")
from project_utils import load_config

pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 100)

# Load Results

Also format for `evaluate_generations.py`

In [2]:
results_dir = f"{os.environ['MDS']}/results_mayarr"
experiments_config = load_config(f"{os.environ['MDS']}/code/experiments_mayarr.json")


# Clean score field
def clean_score(score: object):
    try:
        score = int(score)
    except ValueError as err:
        print(f"Issue converting '{score}' to integer")
        score =  np.nan
    return score


class CustomJSONEncoder(json.JSONEncoder):
  """Custom JSON Encoder to handle various datatypes."""

  def default(self, o) -> Any:
    if isinstance(o, np.integer):
      return int(o)
    if isinstance(o, np.floating):
      return float(o)
    if isinstance(o, (np.ndarray, pd.core.arrays.numeric.NumericArray)):
      return o.tolist()
    try:
        if pd.isna(o):
            return None
    except ValueError as err:
        raise ValueError(f"{o=}\n{err=}")
    try:
      sanitized_o = super(CustomJSONEncoder, self).default(o)
    except TypeError as e:
      raise TypeError(f"Failed to encode '{o}'. Error: {e}")
    return sanitized_o


results_df = []
for dataset in experiments_config["datasets"]:
    for model in experiments_config["models"]:
        for split in experiments_config["splits"]:
            base_path = os.path.join(
                results_dir,
                model,
                dataset,
                split
            )
            individual_scores = {}
            for filename in os.listdir(base_path):
                if "prometheus_eval" not in filename:
                    continue
                
                # Skip non-source and non-reference results
                if ("source" not in filename) or ("ref" not in filename):
                    continue
                dimension = filename.split("_")[-1].strip(".jsonl")
                
                df = pd.read_json(
                    os.path.join(base_path, filename),
                    lines=True
                )
                df["model"] = model
                df["dataset"] = dataset
                df["split"] = split
                df["source_mode"] = "includes source" if "source" in filename else "no source"
                df["reference_mode"] = "includes reference" if "ref" in filename else "no reference"
                df["dimension"] = dimension
                df = df.reset_index().rename(columns={"index": "id"})
                df["score"] = df.score.map(clean_score).astype(dtype='Int64')
                results_df.append(df)
                individual_scores[dimension] = df["score"].values
            
            # Scores are calculated for individual documents. Aggregate them.
            aggregated_scores = {
                "overall": {
                    k: np.nanmean(v) for k, v in individual_scores.items()
                }
            }
            # Re-group the individual scores
            regrouped_scores = []
            for idx in range(len(df)):
                regrouped_scores.append({
                    k: v[idx] for k, v in individual_scores.items()
                })
            aggregated_scores["individual"] = regrouped_scores
    
            output_file = os.path.join(base_path, "prometheus_aggregated.json")
            with open(output_file, "w") as f:
                json.dump(aggregated_scores, f, cls=CustomJSONEncoder)
            print(f"Saved to {output_file}")

results_df = pd.concat(results_df).reset_index(drop=True)

Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/ground_truth/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/allenai/PRIMERA-multixscience/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/PRIMERA-ConvoSumm/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/PRIMERA-MultiNews_Plus/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/SummN-MultiXScience/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results_mayarr/SummN-ConvoSumm/convosumm/reddit/test/prometheus_aggregated.json
Saved to /home/adeluci2/scratch4-mdredze1/aade

# Quantitative

In [ ]:
results_df.groupby(["model", "dataset", "dimension", "source_mode", "reference_mode"]).agg({"score": ["mean", "std"]})


Across dimensions, what is the difference between including source/reference?

In [ ]:
results_df.groupby(["dimension", "source_mode", "reference_mode"]).agg({"score": ["mean", "std"]})


In [ ]:
def filter(row: pd.Series) -> bool:
    if row.source_mode != "includes source":
        return False
    if row.reference_mode != "includes reference":
        return False
    return True

results_df[results_df.apply(filter, axis="columns")].groupby(["dimension", "dataset"]).agg({"score": ["mean", "std"]})


In [ ]:
def map_model_type(model_name: str) -> str:
    model_type = model_name.split("-")[0]
    model_type = model_type.strip("allenai/")
    if model_type == "ICL" and model_name != "ICL":
        model_type = "ICL (few-shot)"
    return model_type

results_df["model_type"] = results_df.model.map(map_model_type)
results_df[results_df.apply(filter, axis="columns")].groupby(["dimension", "model_type"]).agg({"score": ["mean", "std"]})


# Qualitative

How does the score fluctuate for a single example across modes of including the reference and the source documents?

In [ ]:
random.seed(42)
sampled_ids = random.choices(results_df.id.unique(), k=1)

for group_name, df in results_df[results_df.id.isin(sampled_ids)].groupby(["id", "model", "dataset", "dimension"]):
    print(" ".join([str(s) for s in group_name]), "\n")
    for row_id, row in df.iterrows():
        print(f"""PREDICTION: {row.prediction}
REFERENCE: {row.reference}

SCORE: {row.score}. {row.feedback}
MODE: {row.source_mode} and {row.reference_mode}
---------
""")
    print("********************")
    

## Ground Truth

In [ ]:
random.seed(42)
sampled_ids = random.choices(results_df.id.unique(), k=1)

for group_name, df in results_df[results_df.id.isin(sampled_ids)&(results_df.model=="ground_truth")].groupby(["id", "model", "dataset", "dimension"]):
    print(" ".join([str(s) for s in group_name]), "\n")
    for row_id, row in df.iterrows():
        print(f"""PREDICTION: {row.prediction}
REFERENCE: {row.reference}

SCORE: {row.score}. {row.feedback}
MODE: {row.source_mode} and {row.reference_mode}
---------
""")
    print("********************")
 

## Specific Model

In [ ]:
random.seed(42)
sampled_ids = random.choices(results_df.id.unique(), k=1)
model_name = "ICL-convosumm/reddit"

for group_name, df in results_df[results_df.id.isin(sampled_ids)&(results_df.model==model_name)].groupby(["id", "model", "dataset", "dimension"]):
    print(" ".join([str(s) for s in group_name]), "\n")
    for row_id, row in df.iterrows():
        print(f"""PREDICTION: {row.prediction}
        
REFERENCE: {row.reference}

SCORE: {row.score}. {row.feedback}
MODE: {row.source_mode} and {row.reference_mode}
---------
""")
    print("********************")